In [1]:
import os
import time
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from tqdm.auto import tqdm
from PIL import Image, UnidentifiedImageError

# ----------------------------
# Reproducibility
# ----------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

Using device: cpu
PyTorch version: 2.11.0+cpu


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
from pathlib import Path

# Project + data directories
PROJECT_DIR_S = Path('/content/drive/Shareddrives/COS 801 Group 6')
IMG_BASE_DIR = PROJECT_DIR_S / 'data/raw'
LOCAL_SPLIT_CSV = PROJECT_DIR_S / 'data/data_split_resolved.csv'

# Model + results directories
MODEL_DIR = PROJECT_DIR_S / 'models'
RESULTS_DIR = PROJECT_DIR_S / 'results'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Baseline + compression paths (EfficientNet-B0 focus)
BASELINE_MODEL_PATH   = MODEL_DIR / 'effnetb0_baseline_best.pth'
PRUNED_MODEL_PATH     = MODEL_DIR / 'effnetb0_pruned_best.pth'
QUANTISED_MODEL_PATH  = MODEL_DIR / 'effnetb0_pruned_dynamic_quantised_state.pth'
DISTILLED_MODEL_PATH  = MODEL_DIR / 'distilled_effnetb0_student_best.pth'
QUANTISED_STUDENT_PATH = MODEL_DIR / 'distilled_effnetb0_dynamic_quantised_state.pth'

RESULTS_PATH = RESULTS_DIR / 'model_compression_results.csv'

print(f'Image directory: {IMG_BASE_DIR}')
print(f'Model output directory: {MODEL_DIR}')


Image directory: /content/drive/Shareddrives/COS 801 Group 6/data/raw
Model output directory: /content/drive/Shareddrives/COS 801 Group 6/models


In [4]:
import urllib.parse
import os
import pandas as pd
from pathlib import Path

df_split = pd.read_csv(LOCAL_SPLIT_CSV)

# Helper to clean and normalize paths (rather than just filenames)
def normalize_relative_path(p):
    if pd.isna(p):
        return []
    raw_str = str(p).replace('\\', '/').strip().lower()

    # Remove potential redundant 'data/raw/' prefixes
    if raw_str.startswith('data/raw/'):
        raw_str = raw_str.replace('data/raw/', '', 1)

    # Generate multiple candidate decodings to handle '%2B' differences on filesystems
    candidates = []
    # 1. Standard unquoted (decodes %2B to +)
    unquoted = urllib.parse.unquote(raw_str)
    candidates.append(unquoted)

    # 2. Treat '+' as space (common in web queries and some downloaders)
    candidates.append(unquoted.replace('+', ' '))

    # 3. Keep original %2B intact
    candidates.append(raw_str)

    return candidates

# Safely load existing relative paths from disk relative to IMG_BASE_DIR
image_extensions = {'.png', '.jpg', '.jpeg', '.gif', '.bmp', '.tiff', '.webp'}
existing_relative_paths = set()

for root, _, files in os.walk(IMG_BASE_DIR):
    for name in files:
        if Path(name).suffix.lower() in image_extensions:
            full_path = Path(root) / name
            try:
                rel_path = full_path.relative_to(IMG_BASE_DIR)
                existing_relative_paths.add(str(rel_path).replace('\\', '/').lower())
            except ValueError:
                continue

# Build clean lookup sets using paths
existing_normalized_paths = {p for p in existing_relative_paths if p}

# Map each CSV path to whether at least one of its normalized candidate paths exists on disk
matched_count = 0
missing = set()

for p in df_split['resolved_path']:
    if pd.isna(p):
        continue
    candidates = normalize_relative_path(p)
    # Check if any candidate decoding exists in our on-disk files
    if any(cand in existing_normalized_paths for cand in candidates):
        matched_count += 1
    else:
        # Store the primary unquoted candidate as missing for diagnosis
        missing.add(candidates[0])

print(f'Total unique normalized paths on disk: {len(existing_normalized_paths):,}')
print(f'Total records in CSV split: {len(df_split):,}')
print(f'Robust Path Intersection (handling %2B / + / spaces): {matched_count:,}')
print(f'Still missing: {len(missing):,}')

# Diagnostics: Show details on missing paths if any exist
if len(missing) > 0:
    print("\n--- Unmatched Paths Diagnostic Sample (Post-Fix) ---")
    missing_sample = list(missing)[:10]
    for m in missing_sample:
        print(f"CSV Relative Path: '{m}' -> Not found under {IMG_BASE_DIR}")

Total unique normalized paths on disk: 165,488
Total records in CSV split: 38,368
Robust Path Intersection (handling %2B / + / spaces): 38,368
Still missing: 0


In [5]:
#python shared_class_labels.py
#helper script for mapping shared classes between the plantvillage and plantdoc datasets
import json
import unicodedata
import urllib.parse
from pathlib import Path

# canonical -> (PlantVillage folder, PlantDoc folder)
SHARED_CLASSES: dict[str, tuple[str, str]] = {
    "apple_scab": ("Apple___Apple_scab", "Apple Scab Leaf"),
    "apple_rust": ("Apple___Cedar_apple_rust", "Apple rust leaf"),
    "apple_healthy": ("Apple___healthy", "Apple leaf"),
    "blueberry_healthy": ("Blueberry___healthy", "Blueberry leaf"),
    "cherry_healthy": ("Cherry_(including_sour)___healthy", "Cherry leaf"),
    "corn_gray_spot": ("Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot", "Corn Gray leaf spot"),
    "corn_rust": ("Corn_(maize)___Common_rust_", "Corn rust leaf"),
    "corn_blight": ("Corn_(maize)___Northern_Leaf_Blight", "Corn leaf blight"),
    "grape_black_rot": ("Grape___Black_rot", "grape leaf black rot"),
    "grape_healthy": ("Grape___healthy", "grape leaf"),
    "peach_healthy": ("Peach___healthy", "Peach leaf"),
    "pepper_healthy": ("Pepper,_bell___healthy", "Bell_pepper leaf"),
    "potato_early_blight": ("Potato___Early_blight", "Potato leaf early blight"),
    "potato_late_blight": ("Potato___Late_blight", "Potato leaf late blight"),
    "raspberry_healthy": ("Raspberry___healthy", "Raspberry leaf"),
    "soybean_healthy": ("Soybean___healthy", "Soyabean leaf"),
    "squash_mildew": ("Squash___Powdery_mildew", "Squash Powdery mildew leaf"),
    "strawberry_healthy": ("Strawberry___healthy", "Strawberry leaf"),
    "tomato_bacterial_spot": ("Tomato___Bacterial_spot", "Tomato leaf bacterial spot"),
    "tomato_early_blight": ("Tomato___Early_blight", "Tomato Early blight leaf"),
    "tomato_late_blight": ("Tomato___Late_blight", "Tomato leaf late blight"),
    "tomato_leaf_mold": ("Tomato___Leaf_Mold", "Tomato mold leaf"),
    "tomato_septoria": ("Tomato___Septoria_leaf_spot", "Tomato Septoria leaf spot"),
    "tomato_yellow_virus": ("Tomato___Tomato_Yellow_Leaf_Curl_Virus", "Tomato leaf yellow virus"),
    "tomato_mosaic_virus": ("Tomato___Tomato_mosaic_virus", "Tomato leaf mosaic virus"),
    "tomato_healthy": ("Tomato___healthy", "Tomato leaf"),
}

CLASS_NAMES = tuple(SHARED_CLASSES)
N_CLASSES = len(CLASS_NAMES)
CLASS_INDEX = {name: i for i, name in enumerate(CLASS_NAMES)}

def _norm(name):
    # Unquote URL sequences (e.g. %2B to +) and decode plus symbols/spaces robustly
    decoded = urllib.parse.unquote(str(name)).replace('+', ' ').strip()
    return " ".join(unicodedata.normalize("NFKC", decoded).split()).casefold()

_LOOKUP = {
    "lab":   {_norm(pv): c for c, (pv, _) in SHARED_CLASSES.items()},
    "field": {_norm(pd): c for c, (_, pd) in SHARED_CLASSES.items()},
}
assert len(_LOOKUP["lab"]) == N_CLASSES, "a PlantVillage folder is mapped twice"
assert len(_LOOKUP["field"]) == N_CLASSES, "a PlantDoc folder is mapped twice"

def to_canonical(folder_name, domain):
    return _LOOKUP[domain].get(_norm(folder_name))

def to_index(folder_name, domain):
    name = to_canonical(folder_name, domain)
    return None if name is None else CLASS_INDEX[name]

# Linked directly to user's Google Drive raw data directory
DATA = Path('/content/drive/Shareddrives/COS 801 Group 6/data/raw')
LAB_ROOT = [
    DATA / "plantvillage" / "plantvillage dataset" / "color",
    DATA / "plantvillage" / "plantvillage dataset" / "grayscale",
    DATA / "plantvillage" / "plantvillage dataset" / "segmented"
]
FIELD_ROOTS = [DATA / "plantdoc" / "train", DATA / "plantdoc" / "test"]

IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def _subfolders(path):
    return {_norm(p.name) for p in path.iterdir() if p.is_dir()}

def _check(root, domain):
    root = Path(root)
    if not root.is_dir():
        existing = root
        while not existing.is_dir() and existing != existing.parent:
            existing = existing.parent
        found = sorted(p.name for p in existing.iterdir() if p.is_dir())[:20]
        raise FileNotFoundError(
            f"{domain} path does not exist:\n  {root}\n"
            f"Deepest folder that does exist:\n  {existing}\n"
            f"It contains: {found or 'no subfolders'}\n"
            f"Fix the paths at the top of section 2."
        )
    return root

def _collect(root, domain):
    pairs, skipped = [], {}
    for folder in sorted(p for p in root.iterdir() if p.is_dir()):
        images = [p for p in sorted(folder.rglob("*"))
                  if p.suffix.lower() in IMAGE_SUFFIXES]
        label = to_index(folder.name, domain)
        if label is None:
            skipped[folder.name] = skipped.get(folder.name, 0) + len(images)
        else:
            pairs += [(p, label) for p in images]
    return pairs, skipped

def build_index(lab_root=LAB_ROOT, field_roots=FIELD_ROOTS):
    if isinstance(lab_root, (str, Path)):
        lab_roots = [_check(lab_root, "lab")]
    else:
        lab_roots = [_check(root, "lab") for root in lab_root]

    field_roots = [_check(root, "field") for root in field_roots]

    lab = []
    skipped = {}
    for root in lab_roots:
        pairs, also_skipped = _collect(root, "lab")
        lab += pairs
        for name, n in also_skipped.items():
            skipped[name] = skipped.get(name, 0) + n

    field = []
    for root in field_roots:
        pairs, also_skipped = _collect(root, "field")
        field += pairs
        for name, n in also_skipped.items():
            skipped[name] = skipped.get(name, 0) + n

    lab_folders = set()
    for root in lab_roots:
        lab_folders |= _subfolders(root)

    field_folders = set()
    for root in field_roots:
        field_folders |= _subfolders(root)

    missing = []
    for pv, pd in SHARED_CLASSES.values():
        if _norm(pv) not in lab_folders:
            missing.append(pv)
        if _norm(pd) not in field_folders:
            missing.append(pd)

    return {
        "lab_root": lab_roots,
        "field_roots": field_roots,
        "lab": lab,
        "field": field,
        "skipped": skipped,
        "missing": missing,
    }

def count_per_class(pairs):
    counts = [0] * N_CLASSES
    for _, label in pairs:
        counts[label] += 1
    return counts

def report(index, min_field=15, min_lab=50):
    """Print the counts and the warnings. Returns the number of problems."""
    for n, root in enumerate(index['lab_root']):
        print(f"{'lab   root    ' if n == 0 else '              '}{root}")
    for n, root in enumerate(index["field_roots"]):
        print(f"{'field root    ' if n == 0 else '              '}{root}")
    print()

    lab = count_per_class(index["lab"])
    field = count_per_class(index["field"])
    w = max(len(name) for name in CLASS_NAMES)

    print(f"{'class':<{w}}  {'lab':>7}  {'field':>6}")
    print("-" * (w + 17))
    for i, name in enumerate(CLASS_NAMES):
        if lab[i] == 0 or field[i] == 0:
            note = "  <- EMPTY"
        elif field[i] < min_field:
            note = "  <- thin field class"
        elif lab[i] < min_lab:
            note = "  <- thin lab class"
        else:
            note = ""
        print(f" {name:<{w}}  {lab[i]:>7}  {field[i]:>6}{note}")
    print("-" * (w + 17))
    print(f"{'total':<{w}}  {sum(lab):>7}  {sum(field):>6}")
    print(f"\n{N_CLASSES} classes.")

    skipped = index["skipped"]
    if skipped:
        total = sum(skipped.values())
        print(f"\nNot in the map — {total} images in {len(skipped)} folders:")
        for name, n in sorted(skipped.items(), key=lambda item: -item[1]):
            print(f"  {name}  ({n})")
        print("  Any of these that could pair across the two datasets needs a")
        print("  reason recorded in the exclusions comment above.")

    thin = [CLASS_NAMES[i] for i in range(N_CLASSES) if 0 < field[i] < min_field]
    if thin:
        print(f"\nUnder {min_field} field images: {', '.join(thin)}")
        print("  Macro-F1 weights these the same as the largest class. Either drop")
        print("  them and say so, or keep them and state the per-class noise floor.")

    problems = [f"folder in the map but not on disk: {name!r}"
                for name in index["missing"]]
    problems += [f"{CLASS_NAMES[i]}: no images on one side"
                 for i in range(N_CLASSES) if lab[i] == 0 or field[i] == 0]
    if problems:
        print(f"\n{len(problems)} PROBLEM(S) — do not train through these:")
        for line in problems:
            print(f"  - {line}")
    else:
        print("\nChecks passed.")
    return len(problems)

def save_classes(path="classes.json"):
    Path(path).write_text(json.dumps({
        "n_classes": N_CLASSES,
        "class_names": list(CLASS_NAMES),
        "plantvillage": {k: v[0] for k, v in SHARED_CLASSES.items()},
        "plantdoc": {k: v[1] for k, v in SHARED_CLASSES.items()},
    }, indent=2))

if __name__ == "__main__":
    problems = report(build_index())

lab   root    /content/drive/Shareddrives/COS 801 Group 6/data/raw/plantvillage/plantvillage dataset/color
              /content/drive/Shareddrives/COS 801 Group 6/data/raw/plantvillage/plantvillage dataset/grayscale
              /content/drive/Shareddrives/COS 801 Group 6/data/raw/plantvillage/plantvillage dataset/segmented
field root    /content/drive/Shareddrives/COS 801 Group 6/data/raw/plantdoc/train
              /content/drive/Shareddrives/COS 801 Group 6/data/raw/plantdoc/test

class                      lab   field
--------------------------------------
 apple_scab                1890      93
 apple_rust                 825      88
 apple_healthy             4935      91
 blueberry_healthy         4506     115
 cherry_healthy            2562      57
 corn_gray_spot            1539      68
 corn_rust                 3576     116
 corn_blight               2955     191
 grape_black_rot           3540      64
 grape_healthy             1269      69
 peach_healthy             10

In [6]:
train_df = df_split.loc[df_split['split'].eq('train')].copy().reset_index(drop=True)
val_df = df_split.loc[df_split['split'].eq('val')].copy().reset_index(drop=True)
test_df = df_split.loc[df_split['split'].eq('test')].copy().reset_index(drop=True)
field_test_df = df_split.loc[df_split['split'].eq('field_test')].copy().reset_index(drop=True)

print(f'Training samples: {len(train_df):,}')
print(f'Validation samples: {len(val_df):,}')
print(f'Test samples: {len(test_df):,}')
print(f'Field-test samples: {len(field_test_df):,}')

if train_df.empty or val_df.empty or test_df.empty:
    raise RuntimeError('Train, validation and test splits must all contain at least one image.')

# Enforce the official canonical classes and ordering from the CLASS_NAMES script
class_to_idx = CLASS_INDEX.copy()
idx_to_class = {index: class_name for class_name, index in class_to_idx.items()}
N_CLASSES = len(class_to_idx)

print(f'\nNumber of classes (aligned with script): {N_CLASSES}')
print('Enforced mapping index:')
print(class_to_idx)

Training samples: 25,108
Validation samples: 5,385
Test samples: 5,376
Field-test samples: 2,499

Number of classes (aligned with script): 26
Enforced mapping index:
{'apple_scab': 0, 'apple_rust': 1, 'apple_healthy': 2, 'blueberry_healthy': 3, 'cherry_healthy': 4, 'corn_gray_spot': 5, 'corn_rust': 6, 'corn_blight': 7, 'grape_black_rot': 8, 'grape_healthy': 9, 'peach_healthy': 10, 'pepper_healthy': 11, 'potato_early_blight': 12, 'potato_late_blight': 13, 'raspberry_healthy': 14, 'soybean_healthy': 15, 'squash_mildew': 16, 'strawberry_healthy': 17, 'tomato_bacterial_spot': 18, 'tomato_early_blight': 19, 'tomato_late_blight': 20, 'tomato_leaf_mold': 21, 'tomato_septoria': 22, 'tomato_yellow_virus': 23, 'tomato_mosaic_virus': 24, 'tomato_healthy': 25}


In [7]:
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 2

train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_test_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [8]:
from torch.utils.data import Dataset, DataLoader
import urllib.parse
import random
from pathlib import Path

class PlantVillageDataset(Dataset):
    def __init__(self, dataframe, transform=None, class_to_idx=None, raw_dir=IMG_BASE_DIR):
        self.dataframe = dataframe.reset_index(drop=True).copy()
        self.transform = transform
        self.class_to_idx = class_to_idx or {}
        self.raw_dir = Path(raw_dir)

    def __len__(self):
        return len(self.dataframe)

    def _get_single_item(self, idx):
        row = self.dataframe.iloc[idx]
        rel_path = row['path']
        if rel_path.startswith('data/raw/'):
            rel_path = rel_path.replace('data/raw/', '', 1)

        # Generate path candidates to handle URL escaping issues like '%2B', '+', and spaces
        rel_path_str = str(rel_path).replace('\\', '/')
        candidates = [
            rel_path_str,
            urllib.parse.unquote(rel_path_str),
            urllib.parse.unquote(rel_path_str).replace('+', ' ')
        ]

        img_path = None
        for cand in candidates:
            candidate_path = self.raw_dir / cand
            if candidate_path.exists():
                img_path = candidate_path
                break

        # If no candidates exist, fall back to the raw path so we raise an appropriate FileNotFoundError
        if img_path is None:
            img_path = self.raw_dir / rel_path

        try:
            image = Image.open(img_path).convert('RGB')
        except (FileNotFoundError, UnidentifiedImageError, OSError) as exc:
            raise RuntimeError(f'Could not load image: {img_path}') from exc

        label = row['class']
        if label not in self.class_to_idx:
            raise KeyError(f'Class {label} was not found in class_to_idx.')
        label_idx = self.class_to_idx[label]

        if self.transform:
            image = self.transform(image)

        return image, label_idx

    def __getitem__(self, idx):
        try:
            return self._get_single_item(idx)
        except Exception as e:
            print(f"\n[Dataset Warning] Skipping corrupt or missing image index {idx}: {e}")
            # Fallback: keep trying random indices until a valid image is loaded
            for _ in range(100):
                random_idx = random.randint(0, len(self.dataframe) - 1)
                try:
                    return self._get_single_item(random_idx)
                except Exception:
                    continue
            raise e

# 1. Instantiate the Datasets with correct casing
train_dataset = PlantVillageDataset(train_df, train_transforms, class_to_idx)
val_dataset = PlantVillageDataset(val_df, val_test_transforms, class_to_idx)
test_dataset = PlantVillageDataset(test_df, val_test_transforms, class_to_idx)
field_test_dataset = PlantVillageDataset(field_test_df, val_test_transforms, class_to_idx)

# Configure loaders
loader_kwargs = {
    'batch_size': BATCH_SIZE,
    'num_workers': NUM_WORKERS,
    'pin_memory': torch.cuda.is_available(),
    'persistent_workers': NUM_WORKERS > 0,
}

train_loader = DataLoader(train_dataset, shuffle=True, **loader_kwargs)
val_loader = DataLoader(val_dataset, shuffle=False, **loader_kwargs)
test_loader = DataLoader(test_dataset, shuffle=False, **loader_kwargs)
field_test_loader = DataLoader(field_test_dataset, shuffle=False, **loader_kwargs)

print(f"DataLoaders prepared successfully with corruption-handling & %2B resolving! Check batch count: {len(train_loader)}")

DataLoaders prepared successfully with corruption-handling & %2B resolving! Check batch count: 785


In [9]:
# 2. Define the missing training loop and epoch runners
def train_one_epoch(model, data_loader, criterion, optimizer, device, description='Training'):
    model.train()
    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    for inputs, labels in tqdm(data_loader, desc=description, leave=False):
        inputs = inputs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)
        running_loss += loss.item() * batch_size
        correct_predictions += outputs.argmax(dim=1).eq(labels).sum().item()
        total_samples += batch_size

    return running_loss / total_samples, correct_predictions / total_samples


def evaluate_model(model, data_loader, criterion, device, description='Evaluation'):
    if len(data_loader.dataset) == 0:
        return float('nan'), float('nan')

    model.eval()
    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    with torch.inference_mode():
        for inputs, labels in tqdm(data_loader, desc=description, leave=False):
            inputs = inputs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = model(inputs)
            loss = criterion(outputs, labels)

            batch_size = labels.size(0)
            running_loss += loss.item() * batch_size
            correct_predictions += outputs.argmax(dim=1).eq(labels).sum().item()
            total_samples += batch_size

    return running_loss / total_samples, correct_predictions / total_samples


def fit_model_with_scheduler(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    scheduler,
    device,
    epochs,
    checkpoint_path,
    extra_checkpoint_data=None,
):
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    best_val_acc = -1.0
    best_state_dict = None

    for epoch in range(epochs):
        print(f'\nEpoch {epoch + 1}/{epochs}')

        train_loss, train_acc = train_one_epoch(
            model, train_loader, criterion, optimizer, device,
            description=f'Train {epoch + 1}/{epochs}'
        )
        val_loss, val_acc = evaluate_model(
            model, val_loader, criterion, device,
            description=f'Validation {epoch + 1}/{epochs}'
        )

        scheduler.step()

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        print(f'Train loss: {train_loss:.4f} | Train accuracy: {train_acc:.4f}')
        print(f'Val loss:   {val_loss:.4f} | Val accuracy:   {val_acc:.4f}')

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state_dict = copy.deepcopy(model.state_dict())

            checkpoint = {
                'model_state_dict': best_state_dict,
                'class_to_idx': class_to_idx,
                'num_classes': N_CLASSES,
                'validation_accuracy': best_val_acc,
                'epoch': epoch + 1,
            }
            if extra_checkpoint_data:
                checkpoint.update(extra_checkpoint_data)

            torch.save(checkpoint, checkpoint_path)
            print(f'Best checkpoint saved to: {checkpoint_path}')

    if best_state_dict is not None:
        model.load_state_dict(best_state_dict)
    return model, history, best_val_acc

In [10]:
## Model Initialization & Setup
## Model Initialization & Setup
def build_effnetb0(num_classes, use_pretrained_weights=True):
    weights = EfficientNet_B0_Weights.IMAGENET1K_V1 if use_pretrained_weights else None
    model = efficientnet_b0(weights=weights)
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)
    return model

print("Instantiating EfficientNet-B0 baseline model...")
model = build_effnetb0(N_CLASSES, use_pretrained_weights=True).to(device)

# Fixed to 5 epochs regardless of GPU/CPU
BASELINE_EPOCHS = 5
print("Training baseline for 5 epochs.")

BASELINE_LR = 1e-3
criterion = nn.CrossEntropyLoss()
baseline_optimizer = optim.AdamW(model.parameters(), lr=BASELINE_LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(baseline_optimizer, T_max=BASELINE_EPOCHS)

Instantiating EfficientNet-B0 baseline model...
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 113MB/s] 


Training baseline for 5 epochs.


In [ ]:
model, baseline_history, best_baseline_val_acc = fit_model_with_scheduler(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=baseline_optimizer,
    scheduler=scheduler,
    device=device,
    epochs=BASELINE_EPOCHS,
    checkpoint_path=BASELINE_MODEL_PATH,
    extra_checkpoint_data={
        'architecture': 'efficientnet_b0',
        'image_size': IMG_SIZE,
    },
)

print(f'\nBest baseline validation accuracy: {best_baseline_val_acc:.4f}')



Epoch 1/5


Train 1/5:   0%|          | 0/785 [00:00<?, ?it/s]

In [ ]:
import matplotlib.pyplot as plt

def plot_training_history(history, title_prefix='Model'):
    epochs = range(1, len(history['train_loss']) + 1)
    plt.figure(figsize=(12, 4))

    # Loss curves
    plt.subplot(1, 2, 1)
    plt.plot(epochs, history['train_loss'], label='Training loss')
    plt.plot(epochs, history['val_loss'], label='Validation loss')
    plt.title(f'{title_prefix}: Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)

    # Accuracy curves
    plt.subplot(1, 2, 2)
    plt.plot(epochs, history['train_acc'], label='Training accuracy')
    plt.plot(epochs, history['val_acc'], label='Validation accuracy')
    plt.title(f'{title_prefix}: Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.show()

# Updated call for EfficientNet-B0 baseline
plot_training_history(baseline_history, 'EfficientNet-B0 baseline')


In [ ]:
# Evaluate baseline model on lab test split
baseline_test_loss, baseline_test_acc = evaluate_model(
    model, test_loader, criterion, device, description='EfficientNet-B0 baseline test'
)
print(f'EfficientNet-B0 baseline test loss: {baseline_test_loss:.4f}')
print(f'EfficientNet-B0 baseline test accuracy: {baseline_test_acc:.4f}')

# Evaluate baseline model on field test split (if available)
if len(field_test_dataset) > 0:
    baseline_field_loss, baseline_field_acc = evaluate_model(
        model, field_test_loader, criterion, device, description='EfficientNet-B0 baseline field test'
    )
    print(f'EfficientNet-B0 baseline field-test loss: {baseline_field_loss:.4f}')
    print(f'EfficientNet-B0 baseline field-test accuracy: {baseline_field_acc:.4f}')
else:
    print('Field-test evaluation skipped because the field-test split is empty.')
